# Caso 07: Inspecciones sanitarias de Chicago

## Integrantes:
- Graciela Gómez
- Benjamín Gutiérrez
- Juan Pablo Mondaca
- Benjamín Vargas

# 1. Carga base de datos

Todos los comentarios respecto a la decisión del filtrado se encuentran en la documentación.

In [17]:
import pandas as pd
import numpy as np

# Cambia el nombre por el de tu archivo
datos = pd.read_csv("datos.csv", dtype=str)

print(datos.shape)
datos.head()


(104468, 17)


,inspection_id,dba_name,aka_name,license_,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2243957,TWO MILE COFFEE BAR,TWO MILE COFEE BAR,2641611,Restaurant,Risk 2 (Medium),1766 W 95TH ST,CHICAGO,IL,60643,2019-01-02T00:00:00.000,License,Not Ready,NaN,41.72126016238349,-87.66725709711507,"(41.72126016238349, -87.66725709711507)"
1,2243958,FAMILY DOLLART #32124,FAMILY DOLLAR,2630979,Grocery Store,Risk 3 (Low),1633 W MONTEREY AVE,CHICAGO,IL,60643,2019-01-02T00:00:00.000,License,Not Ready,NaN,41.689431926833365,-87.66301447393185,"(41.689431926833365, -87.66301447393185)"
2,2243959,CARMELO'S TACO PLACE,CARMELO'S TACO PLACE,2617376,Restaurant,Risk 1 (High),2746 W 59TH ST,CHICAGO,IL,60629,2019-01-02T00:00:00.000,License,Not Ready,NaN,41.78652776994852,-87.69303014768134,"(41.78652776994852, -87.69303014768134)"
3,2243960,AVONDALE TAP,AVONDALE TAP,2368771,Restaurant,Risk 1 (High),3634 W BELMONT AVE,CHICAGO,IL,60618,2019-01-02T00:00:00.000,Canvass,No Entry,NaN,41.939307518988336,-87.71873540202452,"(41.939307518988336, -87.71873540202452)"
4,2243961,MR. POLLO,MR. POLLO,2542966,Restaurant,Risk 1 (High),3026 W BELMONT AVE,CHICAGO,IL,60618,2019-01-02T00:00:00.000,Canvass,Pass w/ Conditions,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.93946745056518,-87.70381061827938,"(41.93946745056518, -87.70381061827938)"


# 2. Unificación y limpieza de variables

Todos los supuestos y consideraciones asumidas se encuentran en la documentación.

In [18]:
# Bloque para definición de funciones:
def limpiar_texto(x):
    return (
        x.astype("string").str.upper().str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA))

# Convierte una columna a texto, pasa sus valores a mayúsculas,
# elimina espacios sobrantes y reemplaza los textos vacíos por NA.

def homologar_ciudad(x):
    ciudad = limpiar_texto(x)
    variantes_chicago = ["312CHICAGO","CCHICAGO","CH","CHICAGO",
        "CHICAGO.","CHICAGOBEDFORD PARK","CHICAGOC","CHICAGOCHICAGO","CHICAGOO"]

    return ciudad.mask(
        ciudad.isin(variantes_chicago),
        "CHICAGO")

# Limpia el texto y unifica las variantes de Chicago
# que definimos en nuestras reglas.

def obtener_riesgo_del_dia(x):
    valores = x.dropna().unique()
    if len(valores) == 1:
        return valores[0]
    return pd.NA


In [19]:
# Aplicar la función anterior a todas las variables de interes
datos["nombre"] = limpiar_texto(datos["dba_name"])
datos["nombre_aka"] = limpiar_texto(datos["aka_name"])
datos["tipo_establecimiento"] = limpiar_texto(datos["facility_type"])
datos["direccion"] = limpiar_texto(datos["address"])

# Sobre la variable licencia debemos tener dos posibles consideraciones con respecto a su escritura
datos["licencia"] = limpiar_texto(datos["license_"])
datos["licencia"] = datos["licencia"].str.replace(r"\.0+$","",regex=True)
datos["licencia"] = datos["licencia"].where(
    datos["licencia"].str.fullmatch(r"[0-9]+", na=False) & datos["licencia"].str.contains(r"[1-9]", na=False),pd.NA)

# Sobre city aplicamos su función de homologación
datos["ciudad"] = homologar_ciudad(datos["city"])

# Normalizamos la fecha de inspección
datos["inspection_date"] = pd.to_datetime(
    datos["inspection_date"],errors="raise").dt.normalize()

# Consideraciones con respecto a risk
riesgos_validos = ["Risk 1 (High)","Risk 2 (Medium)","Risk 3 (Low)", "All"]
datos["risk"] = datos["risk"].where(datos["risk"].isin(riesgos_validos),pd.NA)



## 2.1 Identificar único de cada local

In [20]:
datos["id_local"] = (
    datos["licencia"] + " | " + datos["direccion"]).where(
    datos["licencia"].notna() & datos["direccion"].notna(),pd.NA)

locales_unicos = (
    datos.loc[datos["id_local"].notna(),["id_local", "licencia", "direccion"]]
    .drop_duplicates()
    .reset_index(drop=True))

print("Locales distintos:", datos["id_local"].nunique())
print("Inspecciones sin identificación:", datos["id_local"].isna().sum())

Locales distintos: 23705
Inspecciones sin identificación: 252


## 2.2 Conservar *solo* observaciones pertenecientes a Illinois, Chicago.

In [21]:
datos = datos.loc[datos["state"].eq("IL").fillna(False)].copy()
datos = datos.loc[datos["ciudad"].eq("CHICAGO").fillna(False)].copy()

## 2.3 Eliminar variables

Consideraciones para su eliminación se encuentran en la documentación.

In [22]:
datos = datos.drop(
    columns=[
        "dba_name",
        "aka_name",
        "address",
        "license_",
        "facility_type",
        "latitude",
        "longitude",
        "location",
        "ciudad",
        "city",
        "state",
        "zip"
    ],
    errors="ignore"
)
datos = datos.reset_index(drop=True)

In [23]:
# Comprobación de dimensiones:
print("Dimensión de datos:", datos.shape)

Dimensión de datos: (104311, 12)


# 3. CREACIÓN DE PREDICTORES

Los supuestos utilizados para su creación esta informados en la documentación

## 3.1 Predictores (Categoricos)

### Tipo de Establecimiento (facility_type)

In [24]:
filas_iniciales = len(datos)

# TIPO DE ESTABLECIMIENTO:
frecuencias = (
    datos["tipo_establecimiento"]
    .value_counts()
    .rename_axis("tipo_establecimiento")
    .reset_index(name="n")
    .sort_values(
        ["n", "tipo_establecimiento"],
        ascending=[False, True])
    .reset_index(drop=True))

n_cat = 10 # Seleccionar la cantidad de categorías más frecuentes que queremos considerar
print(frecuencias.head(n_cat)) # Categorías con mayor frecuencia


total_con_tipo = frecuencias["n"].sum()
porcentaje_cubierto = (
    frecuencias.head(n_cat)["n"].sum() / total_con_tipo * 100
    if total_con_tipo > 0
    else float("nan"))
print(f"Porcentaje cubierto: {porcentaje_cubierto:.2f}%")

top = frecuencias.head(n_cat)["tipo_establecimiento"].tolist()
datos["tipo_establecimiento"] = (
    datos["tipo_establecimiento"].astype("string")
)

# Caso 1: los tipos conocidos que no pertenecen al top pasan a OTROS.
datos.loc[
    datos["tipo_establecimiento"].notna()
    & ~datos["tipo_establecimiento"].isin(top),
    "tipo_establecimiento"
] = "OTROS"

# Caso 2: los tipos faltantes pasan a SIN INFORMACION.
# Los tipos que pertenecían al top conservan su nombre.
datos["tipo_establecimiento"] = (
    datos["tipo_establecimiento"]
    .fillna("SIN INFORMACION")
    .astype("category")
)
datos["tipo_establecimiento"].value_counts(dropna=False)

              tipo_establecimiento      n
0                       RESTAURANT  72330
1                    GROCERY STORE  11405
2                           SCHOOL   6648
3     CHILDREN'S SERVICES FACILITY   3575
4  DAYCARE ABOVE AND UNDER 2 YEARS   1582
5                           BAKERY   1476
6                   LONG TERM CARE    953
7            DAYCARE (2 - 6 YEARS)    722
8                         CATERING    703
9             MOBILE FOOD PREPARER    488
Porcentaje cubierto: 96.21%


tipo_establecimiento
RESTAURANT                         72330
GROCERY STORE                      11405
SCHOOL                              6648
OTROS                               3937
CHILDREN'S SERVICES FACILITY        3575
DAYCARE ABOVE AND UNDER 2 YEARS     1582
BAKERY                              1476
LONG TERM CARE                       953
DAYCARE (2 - 6 YEARS)                722
CATERING                             703
SIN INFORMACION                      492
MOBILE FOOD PREPARER                 488
Name: count, dtype: int64

### Riesgo (risk)

In [25]:
# RIESGO:
datos = datos.rename(columns={"risk": "riesgo"})

codigos_riesgo = {
    "Risk 1 (High)": 1,
    "Risk 2 (Medium)": 2,
    "Risk 3 (Low)": 3,
    "All": 4,
}


def resumir_riesgo_del_dia(riesgos):
    codigos = riesgos.map(codigos_riesgo)

    # si hay riesgos conocidos, conservar el más alto
    riesgos_conocidos = codigos[codigos.isin([1, 2, 3])]

    if not riesgos_conocidos.empty:
        return int(riesgos_conocidos.min())

    # conservar All si no hay un riesgo conocido ese día.
    if codigos.eq(4).any():
        return 4

    # Todos los registros del día carecen de riesgo.
    return 0

historial_riesgo = (
    datos.loc[
        datos["id_local"].notna()
        & datos["inspection_date"].notna()
    ]
    .groupby(["id_local", "inspection_date"], as_index=False)
    .agg(riesgo_del_dia=("riesgo", resumir_riesgo_del_dia))
    .sort_values(["id_local", "inspection_date"])
    .reset_index(drop=True)
)

historial_riesgo["riesgo_previo"] = (
    historial_riesgo
    .groupby("id_local")["riesgo_del_dia"]
    .shift(1)
    .fillna(0)
    .astype("Int64")
)

# Incorporar el antecedente a cada inspección.
datos = datos.merge(
    historial_riesgo[
        ["id_local", "inspection_date", "riesgo_previo"]
    ],
    on=["id_local", "inspection_date"],
    how="left",
    validate="many_to_one",
)

# Sin antecedentes o sin identificación del local: código 0.
datos["riesgo_previo"] = (
    datos["riesgo_previo"]
    .fillna(0)
    .astype("Int64")
    .astype("category")
)

datos = datos.drop(columns=["riesgo"])

datos["riesgo_previo"].value_counts(dropna=False)

riesgo_previo
1    66265
0    24036
2    11386
3     2611
4       13
Name: count, dtype: int64

### Advertencias (violations)

In [10]:
# ADVERTENCIAS:
infracciones = datos[["inspection_id", "id_local", "inspection_date", "violations"]].copy()

infracciones["violations"] = (infracciones["violations"].astype("string"))
infracciones["sin_texto"] = (infracciones["violations"].isna() | infracciones["violations"].str.strip().eq(""))
infracciones["violations"] = (infracciones["violations"].str.split("|", regex=False))

# Crear una fila por cada fragmento de infracción.
# Esta tabla auxiliar tendrá más filas que datos.
infracciones = infracciones.explode("violations",ignore_index=True)

# LIMPIAR LOS FRAGMENTOS #
infracciones["violations"] = (
    infracciones["violations"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())

infracciones = infracciones.loc[
    infracciones["sin_texto"]
    | (
        infracciones["violations"].notna()
        & infracciones["violations"].ne("")
    )].copy()

# EXTRAER EL CÓDIGO DE INFRACCIÓN #
infracciones["codigo_infraccion"] = (
    infracciones["violations"]
    .str.extract(r"^([0-9]+)(?=\.)", expand=False))

infracciones_por_fecha = (
    infracciones.loc[
        infracciones["id_local"].notna()
        & infracciones["inspection_date"].notna()
    ]
    .groupby(
        ["id_local", "inspection_date"],
        as_index=False
    )
    .agg(
        # True si al menos un fragmento del día carece de código.
        detalle_incompleto=(
            "codigo_infraccion",
            lambda x: x.isna().any()
        ),

        # Contar códigos distintos entre todas las visitas del día.
        n_tipos_infraccion_dia=(
            "codigo_infraccion",
            "nunique"
        )
    )
)

infracciones_por_fecha["n_tipos_infraccion_dia"] = (
    infracciones_por_fecha["n_tipos_infraccion_dia"]
    .astype("Int64")
    .mask(
        infracciones_por_fecha["detalle_incompleto"],
        pd.NA))

infracciones_por_fecha["detalle_incompleto"] = (
    infracciones_por_fecha["detalle_incompleto"]
    .astype("boolean"))

infracciones_por_fecha = (
    infracciones_por_fecha
    .sort_values(["id_local", "inspection_date"])
    .reset_index(drop=True))

# Recuperar la cantidad de códigos de la fecha anterior.
infracciones_por_fecha["n_tipos_infraccion_previos"] = (
    infracciones_por_fecha
    .groupby("id_local")["n_tipos_infraccion_dia"]
    .shift(1))

# Recuperar si el detalle de esa fecha anterior estaba incompleto.
infracciones_por_fecha["detalle_incompleto_previo"] = (
    infracciones_por_fecha
    .groupby("id_local")["detalle_incompleto"]
    .shift(1))

datos = datos.merge(
    infracciones_por_fecha[
        [
            "id_local",
            "inspection_date",
            "n_tipos_infraccion_previos"
            ]
    ],
    on=["id_local", "inspection_date"],
    how="left",
    validate="many_to_one"
)

datos = datos.drop(columns=["violations"])
datos["n_tipos_infraccion_previos"].describe()
datos["n_tipos_infraccion_previos"].isna().sum()


np.int64(41655)

## 3.2 Predictores (Númericos)

In [11]:
# CANTIDAD DE INSPECCIONES POR LOCAL Y FECHA #
visitas_por_fecha = (
    datos.loc[
        datos["id_local"].notna()
        & datos["inspection_date"].notna()
    ]
    .groupby(["id_local", "inspection_date"])
    .size()
    .reset_index(name="visitas_del_dia")
    .sort_values(["id_local", "inspection_date"])
    .reset_index(drop=True))

# ACUMULAR LAS VISITAS ANTERIORES #
visitas_por_fecha["n_visitas_previas"] = (
    visitas_por_fecha
    .groupby("id_local")["visitas_del_dia"]
    .cumsum()
    - visitas_por_fecha["visitas_del_dia"])

datos = datos.merge(
    visitas_por_fecha[
        ["id_local", "inspection_date", "n_visitas_previas"]
    ],
    on=["id_local", "inspection_date"],
    how="left",
    validate="many_to_one")

datos["n_visitas_previas"] = (datos["n_visitas_previas"].astype("Int64"))
datos["historia_no_identificable"] = datos["id_local"].isna()

datos["historia_no_identificable"].value_counts(dropna=False)

historia_no_identificable
False    104061
True        250
Name: count, dtype: int64

In [12]:
################################################################################
######################## CREAR LA BASE DE ANÁLISIS #############################
################################################################################
datos_canvass = datos.loc[
    datos["inspection_type"].eq("Canvass")
    & datos["results"].isin(
        ["Fail", "Pass", "Pass w/ Conditions"]
    )].copy()

filas_canvass = len(datos_canvass)

# BINARIZAR EL RESULTADO #
datos_canvass["resultado"] = (
    datos_canvass["results"]
    .map({
        "Fail": 0,
        "Pass": 1,
        "Pass w/ Conditions": 1
    })
    .astype("Int64"))

datos_canvass = datos_canvass.drop(columns=["results"])
print("Dimensión de datos_canvass:", datos_canvass.shape)


Dimensión de datos_canvass: (43216, 14)


In [13]:
# PROPORCIÓN DE FAILS ENTRE TODAS LAS VISITAS PREVIAS #
historial_resultados = (
    datos.loc[
        datos["id_local"].notna()
        & datos["inspection_date"].notna()
    ]
    .groupby(
        ["id_local", "inspection_date"],
        as_index=False
    )
    .agg(
        # size cuenta filas, aunque inspection_id tenga algún NA.
        visitas_del_dia=("inspection_id", "size"),

        # Contar los resultados Fail.
        # sum() ignora los valores faltantes por defecto.
        fail_del_dia=(
            "results",
            lambda x: x.eq("Fail").sum()
        )
    )
    .sort_values(["id_local", "inspection_date"])
    .reset_index(drop=True))


# ACUMULAR LAS VISITAS PREVIAS #
historial_resultados["n_visitas_previas"] = (
    historial_resultados
    .groupby("id_local")["visitas_del_dia"]
    .cumsum()
    - historial_resultados["visitas_del_dia"])

# ACUMULAR LOS FAILS PREVIOS #
historial_resultados["n_fail_previos"] = (
    historial_resultados
    .groupby("id_local")["fail_del_dia"]
    .cumsum()
    - historial_resultados["fail_del_dia"])


# CALCULAR LA PROPORCIÓN #
historial_resultados["prop_fail_previa"] = (
    historial_resultados["n_fail_previos"]
    / historial_resultados["n_visitas_previas"].where(
        historial_resultados["n_visitas_previas"] > 0
    )
).astype("Float64")

datos_canvass = datos_canvass.merge(
    historial_resultados[
        [
            "id_local",
            "inspection_date",
            "n_fail_previos",
            "prop_fail_previa"
        ]
    ],
    on=["id_local", "inspection_date"],
    how="left",
    validate="many_to_one")

datos_canvass["n_fail_previos"] = (datos_canvass["n_fail_previos"].astype("Int64"))

datos_canvass["n_fail_previos"].describe()
datos_canvass["n_fail_previos"].isna().sum()


np.int64(111)

In [14]:
# TIEMPO DESDE LA ÚLTIMA VISITA ANTERIOR DE CUALQUIER TIPO #
tiempo_entre_visitas = (
    visitas_por_fecha
    .sort_values(["id_local", "inspection_date"])
    .reset_index(drop=True)
    .copy())


# RECUPERAR LA FECHA ANTERIOR #
tiempo_entre_visitas["fecha_ultima_visita"] = (
    tiempo_entre_visitas
    .groupby("id_local")["inspection_date"]
    .shift(1))

# CALCULAR LOS DÍAS TRANSCURRIDOS #
tiempo_entre_visitas["dias_desde_ultima_visita"] = (
    tiempo_entre_visitas["inspection_date"]
    - tiempo_entre_visitas["fecha_ultima_visita"]
).dt.days.astype("Int64")


# CONVERTIR A MESES APROXIMADOS #
tiempo_entre_visitas["meses_desde_ultima_visita"] = (
    tiempo_entre_visitas["dias_desde_ultima_visita"] / 30
)

datos_canvass = datos_canvass.merge(
    tiempo_entre_visitas[
        [
            "id_local",
            "inspection_date",
            "dias_desde_ultima_visita",
            "meses_desde_ultima_visita"
        ]
    ],
    on=["id_local", "inspection_date"],
    how="left",
    validate="many_to_one")

In [15]:
datos_canvass.head()
print(datos_canvass.shape)

(43216, 18)
